# Krones CV Challenge — Train + Infer Evaluation Notebook

This notebook trains and evaluates a **binary ROI classifier** (reusable vs non-reusable bottles) in a **single Run All** — no external weight datasets required. Images are cropped using COCO-style ROI boxes, then classified with a **ConvNeXt-Large** backbone (timm, ImageNet-pretrained) fine-tuned on the competition training set.

We hold out an **8% stratified validation split** to tune the decision threshold (F1-optimal) and apply early stopping. Training uses mixed precision, EMA weights, MixUp/CutMix, and focal loss — matching our local v18 recipe but compressed to fit a **12-hour GPU budget** (~12 epochs, patience 4). **bottletypes.csv** enables stratified splitting diagnostics and per-type error analysis on test predictions.

After training, **best** and **last-epoch** EMA checkpoints are averaged at inference (optional 2-checkpoint ensemble, ~2× infer time) with **test-time augmentation** (4 flip views by default). Both **Performance (F1)** and **Efficiency (wall-clock)** are scored, so `CONFIG` exposes epoch count, TTA views, and model size to trade accuracy for speed within the 12 h limit.

## Environment setup
Install timm and import dependencies. Seeds are fixed; GPU name and memory are printed for verification.

In [ ]:
!pip install -q timm

import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
import torchvision.transforms.v2 as transforms
from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from timm.data import Mixup
from timm.utils import ModelEmaV2
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")

NOTEBOOK_START = time.perf_counter()
MAX_RUNTIME_SEC = None  # set from CONFIG after load

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Device: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GB", flush=True)
else:
    print("Device: CPU (no GPU detected — training will not finish in 12 h)", flush=True)

## Configuration
All paths and hyperparameters in one dict. Update `data_dir` if your Kaggle competition dataset slug differs.

In [ ]:
CONFIG = {
    # --- paths (update slug to match attached competition data) ---
    "data_dir": "/kaggle/input/krones-data/images",
    "checkpoint_path": "/kaggle/working/best_model.pth",
    "last_checkpoint_path": "/kaggle/working/last_model.pth",
    "use_dual_checkpoint_ensemble": True,  # average best + last epoch at infer
    "output_path": "/kaggle/working/submission.csv",
    # --- model / training (12 h budget defaults) ---
    "model_name": "convnext_large",  # convnext_base ~2x faster, slightly lower F1
    "imgsz": 320,
    "batch_size": 4,
    "grad_accum": 4,           # effective batch = 16
    "max_epochs": 12,
    "patience": 4,
    "val_fraction": 0.08,
    "lr": 3e-5,
    "weight_decay": 0.08,
    "warmup_epochs": 2,
    "pretrained": True,        # ImageNet init via timm (no external .pth needed)
    "use_ema": True,
    "use_mixup": True,
    "num_workers": 2,
    # --- inference ---
    "tta_n": 4,                # 4=fast; 8=+rotations (~2x infer time)
    "infer_batch_size": 16,
    "use_bottletypes": True,
    # --- runtime guard ---
    "max_runtime_hours": 12,
    "reserve_infer_hours": 1.5,  # extra margin for 2-checkpoint TTA infer
}

MAX_RUNTIME_SEC = CONFIG["max_runtime_hours"] * 3600
RESERVE_INFER_SEC = CONFIG["reserve_infer_hours"] * 3600

DATA_DIR = Path(CONFIG["data_dir"])
TRAIN_DIR = DATA_DIR / "train_images"
TEST_DIR = DATA_DIR / "test_images"
print(f"Data dir exists: {DATA_DIR.is_dir()}", flush=True)
print(f"Train images: {TRAIN_DIR.is_dir()} | Test images: {TEST_DIR.is_dir()}", flush=True)

## Data loading
Load train/test metadata, ROI boxes from JSON annotations, and optional bottle-type labels.

In [ ]:
def load_all_rois(data_dir: Path) -> dict[str, list[float]]:
    """Merge train + test ROI annotations -> {image_id: [x, y, w, h]}."""
    rois: dict[str, list[float]] = {}
    for fname in ("train_annotations.json", "test_annotations_roi_only.json"):
        path = data_dir / fname
        if not path.is_file():
            print(f"Warning: missing {path}", flush=True)
            continue
        with open(path) as f:
            data = json.load(f)
        id_to_name = {img["id"]: img["file_name"] for img in data.get("images", [])}
        for ann in data.get("annotations", []):
            rois[id_to_name[ann["image_id"]]] = ann["bbox"]
    return rois


def load_bottletypes(data_dir: Path) -> dict[str, str]:
    path = data_dir / "bottletypes.csv"
    if not path.is_file():
        print(f"Warning: bottletypes.csv not found at {path}", flush=True)
        return {}
    df = pd.read_csv(path)
    return dict(zip(df["image_id"], df["bottle_type"]))


def elapsed_hours() -> float:
    return (time.perf_counter() - NOTEBOOK_START) / 3600.0


def remaining_train_budget_sec() -> float:
    used = time.perf_counter() - NOTEBOOK_START
    return MAX_RUNTIME_SEC - RESERVE_INFER_SEC - used


train_df = pd.read_csv(DATA_DIR / "train.csv")
train_df["img_path"] = train_df["image_id"].apply(lambda x: str(TRAIN_DIR / x))
sample_df = pd.read_csv(DATA_DIR / "sample_submission.csv")
rois = load_all_rois(DATA_DIR)
bottletypes = load_bottletypes(DATA_DIR)

n_train_roi = sum(1 for iid in train_df["image_id"] if iid in rois)
n_test_roi = sum(1 for iid in sample_df["image_id"] if iid in rois)
print(f"Train: {len(train_df)} images | ROI coverage: {100*n_train_roi/len(train_df):.1f}%", flush=True)
print(f"Test:  {len(sample_df)} images | ROI coverage: {100*n_test_roi/len(sample_df):.1f}%", flush=True)
print(f"Train positive rate: {100*train_df['target'].mean():.1f}%", flush=True)
if bottletypes:
    n_types = train_df["image_id"].map(bottletypes).nunique(dropna=True)
    print(f"Unique bottle types (train): {n_types}", flush=True)

## Transforms, loss, and threshold helpers
Training augmentations, validation transforms, TTA views, focal loss, and F1 threshold search.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def train_transform(imgsz: int) -> transforms.Compose:
    oversize = int(imgsz * 1.15)
    return transforms.Compose([
        transforms.Resize((oversize, oversize)),
        transforms.RandomCrop(imgsz),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.08),
        transforms.ToImage(),
        transforms.ToDtype(torch.float32, scale=True),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomApply([transforms.RandomPerspective(0.2, p=1.0)], p=0.3),
        transforms.RandomApply([transforms.RandomAutocontrast(p=1.0)], p=0.3),
        transforms.RandomApply([transforms.RandomAdjustSharpness(2.0, p=1.0)], p=0.3),
        transforms.RandomErasing(p=0.3, scale=(0.02, 0.2)),
    ])


def val_transform(imgsz: int) -> transforms.Compose:
    oversize = int(imgsz * 1.08)
    return transforms.Compose([
        transforms.Resize((oversize, oversize)),
        transforms.CenterCrop(imgsz),
        transforms.ToImage(),
        transforms.ToDtype(torch.float32, scale=True),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def tta_views(img: Image.Image, n: int) -> list[Image.Image]:
    w, h = img.size
    c = int(min(w, h) * 0.85)
    left, top = (w - c) // 2, (h - c) // 2
    zoom = img.crop((left, top, left + c, top + c)).resize((w, h), Image.BICUBIC)
    all_views = [
        img,
        img.transpose(Image.FLIP_LEFT_RIGHT),
        img.transpose(Image.FLIP_TOP_BOTTOM),
        img.transpose(Image.FLIP_LEFT_RIGHT).transpose(Image.FLIP_TOP_BOTTOM),
        img.rotate(5, Image.BICUBIC),
        img.rotate(-5, Image.BICUBIC),
        img.rotate(10, Image.BICUBIC),
        img.rotate(-10, Image.BICUBIC),
        img.rotate(90, Image.BICUBIC),
        img.rotate(180, Image.BICUBIC),
        img.rotate(270, Image.BICUBIC),
        zoom,
    ]
    if n not in (4, 8, 12):
        raise ValueError(f"tta_n must be 4, 8, or 12; got {n}")
    return all_views[:n]


class SoftTargetFocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0):
        super().__init__()
        self.gamma = gamma
        self.log_softmax = nn.LogSoftmax(dim=-1)

    def forward(self, inputs, targets):
        log_probs = self.log_softmax(inputs)
        probs = torch.exp(log_probs)
        return (-torch.sum(targets * (1 - probs) ** self.gamma * log_probs, dim=-1)).mean()


def find_best_threshold(y_true, y_probs, steps: int = 199):
    y_true = np.asarray(y_true)
    y_probs = np.asarray(y_probs)
    best_f1, best_t = 0.0, 0.5
    for t in np.linspace(0.01, 0.99, steps):
        score = f1_score(y_true, (y_probs >= t).astype(int), zero_division=0)
        if score > best_f1:
            best_f1, best_t = score, t
    return best_t, best_f1


IMGSZ = CONFIG["imgsz"]
train_tf = train_transform(IMGSZ)
val_tf = val_transform(IMGSZ)
print(f"Transforms ready | imgsz={IMGSZ}", flush=True)

## Dataset classes
ROI-cropped train and test datasets with PyTorch DataLoaders.

In [ ]:
class TrainDS(Dataset):
    def __init__(self, df, rois, transform=None):
        self.df = df.reset_index(drop=True)
        self.rois = rois
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["img_path"]).convert("RGB")
        if row["image_id"] in self.rois:
            x, y, w, h = self.rois[row["image_id"]]
            img = img.crop((x, y, x + w, y + h))
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(row["target"], dtype=torch.long)


class TestDS(Dataset):
    def __init__(self, df, img_dir, rois):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.rois = rois

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img_path = self.img_dir / row["image_id"]
        if not img_path.is_file():
            return None, i
        img = Image.open(img_path).convert("RGB")
        if row["image_id"] in self.rois:
            x, y, w, h = self.rois[row["image_id"]]
            img = img.crop((x, y, x + w, y + h))
        return img, i


def collate_test(batch):
    imgs, indices = [], []
    for img, idx in batch:
        if img is not None:
            imgs.append(img)
            indices.append(idx)
    return imgs, indices


def make_loader(ds, batch_size, shuffle, collate_fn=None):
    kw = {
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": CONFIG["num_workers"],
        "pin_memory": device.type == "cuda",
    }
    if collate_fn is not None:
        kw["collate_fn"] = collate_fn
    if CONFIG["num_workers"] > 0:
        kw["persistent_workers"] = True
        kw["prefetch_factor"] = 2
    return DataLoader(ds, **kw)


split_df, val_df = train_test_split(
    train_df,
    test_size=CONFIG["val_fraction"],
    random_state=SEED,
    stratify=train_df["target"],
)
print(f"Train split: {len(split_df)} | Val split: {len(val_df)}", flush=True)

train_ds = TrainDS(split_df, rois, train_tf)
val_ds = TrainDS(val_df, rois, val_tf)
train_loader = make_loader(train_ds, CONFIG["batch_size"], shuffle=True)
val_loader = make_loader(val_ds, CONFIG["batch_size"] * 2, shuffle=False)
print(f"Train batches/epoch: {len(train_loader)} | Val batches: {len(val_loader)}", flush=True)

## Training loop
Fine-tune ConvNeXt on ROI crops with EMA, AMP, and early stopping. Stops automatically if the 12 h budget (minus inference reserve) is exhausted.

In [ ]:
# RUNTIME BUDGET GUIDE (edit CONFIG to trade F1 vs speed, total <= 12 h):
# convnext_large, 12 ep, tta_n=4, dual ckpt -> ~9-11 h train + ~1 h infer  (default)
# convnext_large,  8 ep, tta_n=4, dual ckpt -> ~6-8 h train + ~1 h infer
# use_dual_checkpoint_ensemble=False     -> ~half infer time, slightly lower F1

TRAIN_START = time.perf_counter()
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats(device)

model_name = CONFIG["model_name"]
model = timm.create_model(
    model_name,
    pretrained=CONFIG["pretrained"],
    num_classes=2,
    drop_rate=0.3,
    drop_path_rate=0.2,
).to(device)

ema = ModelEmaV2(model, decay=0.9998, device=device) if CONFIG["use_ema"] else None
mixup_fn = (
    Mixup(mixup_alpha=0.8, cutmix_alpha=1.0, prob=0.9, switch_prob=0.5,
          mode="batch", label_smoothing=0.1, num_classes=2)
    if CONFIG["use_mixup"]
    else None
)
criterion = SoftTargetFocalLoss(gamma=2.0)
val_criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = SequentialLR(
    optimizer,
    [
        LinearLR(optimizer, start_factor=0.01, total_iters=CONFIG["warmup_epochs"]),
        CosineAnnealingLR(
            optimizer,
            T_max=max(CONFIG["max_epochs"] - CONFIG["warmup_epochs"], 1),
            eta_min=5e-8,
        ),
    ],
    milestones=[CONFIG["warmup_epochs"]],
)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
accum = CONFIG["grad_accum"]
ckpt_path = Path(CONFIG["checkpoint_path"])
last_ckpt_path = Path(CONFIG["last_checkpoint_path"])

best_f1, best_thresh, patience_counter = 0.0, 0.5, 0
best_state = None
best_epoch = 0
last_state = None
last_epoch = 0


@torch.inference_mode()
def validate(eval_model):
    eval_model.eval()
    y_true, y_probs, val_loss = [], [], 0.0
    for x, y in val_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
            out = eval_model(x)
            val_loss += val_criterion(out, y).item()
        y_probs.extend(torch.softmax(out, 1)[:, 1].cpu().numpy())
        y_true.extend(y.cpu().numpy())
    val_loss /= max(len(val_loader), 1)
    thresh, f1 = find_best_threshold(y_true, y_probs)
    return f1, thresh, val_loss


for epoch in range(1, CONFIG["max_epochs"] + 1):
    if remaining_train_budget_sec() <= 0:
        print(f"Stopping training: budget exhausted before epoch {epoch}", flush=True)
        break

    model.train()
    train_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    t0 = time.perf_counter()

    for step, (x, y) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}", leave=True), start=1):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        if mixup_fn is not None:
            x, y = mixup_fn(x, y)
        with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
            loss = criterion(model(x), y) / accum
        scaler.scale(loss).backward()
        if step % accum == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if ema is not None:
                ema.update(model)
        train_loss += loss.item() * accum

    train_loss /= len(train_loader)
    scheduler.step()

    eval_model = ema.module if ema is not None else model
    ep_f1, ep_thresh, val_loss = validate(eval_model)
    last_state = {k: v.cpu().clone() for k, v in eval_model.state_dict().items()}
    last_epoch = epoch
    torch.save(last_state, last_ckpt_path)
    ep_min = (time.perf_counter() - t0) / 60.0
    print(
        f"Epoch {epoch}/{CONFIG['max_epochs']} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
        f"| val F1={ep_f1:.4f} @ thresh={ep_thresh:.4f} | {ep_min:.1f} min/ep | elapsed={elapsed_hours():.2f} h",
        flush=True,
    )

    if ep_f1 > best_f1:
        best_f1, best_thresh, patience_counter = ep_f1, ep_thresh, 0
        best_epoch = epoch
        best_state = {k: v.cpu().clone() for k, v in eval_model.state_dict().items()}
        torch.save(best_state, ckpt_path)
        print(f"  -> New best saved to {ckpt_path}", flush=True)
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["patience"]:
            print(f"Early stop @ epoch {epoch} (patience={CONFIG['patience']})", flush=True)
            break

if best_state is None:
    raise RuntimeError("Training produced no checkpoint — check data paths and GPU.")

threshold = best_thresh
train_hours = (time.perf_counter() - TRAIN_START) / 3600.0
print(f"\nTraining done in {train_hours:.2f} h | best val F1={best_f1:.4f} @ epoch saved", flush=True)
print(f"Checkpoints: best -> {ckpt_path} | last (ep {last_epoch}) -> {last_ckpt_path}", flush=True)
print(f"Threshold (from best checkpoint val): {threshold:.4f}", flush=True)

## Test inference
Load best (+ last epoch if enabled) checkpoints, average TTA probabilities, and apply the validation threshold.

In [ ]:
INFER_START = time.perf_counter()
tta_n = CONFIG["tta_n"]
use_amp = device.type == "cuda"

test_ds = TestDS(sample_df, TEST_DIR, rois)
test_loader = make_loader(
    test_ds, CONFIG["infer_batch_size"], shuffle=False, collate_fn=collate_test
)

infer_checkpoints: list[tuple[str, Path]] = [("best", ckpt_path)]
if CONFIG["use_dual_checkpoint_ensemble"] and last_ckpt_path.is_file() and last_epoch != best_epoch:
    infer_checkpoints.append(("last", last_ckpt_path))
elif CONFIG["use_dual_checkpoint_ensemble"] and last_epoch == best_epoch:
    print("Dual ensemble skipped: last epoch is the same as best epoch.", flush=True)

n_ckpts = len(infer_checkpoints)
print(f"Inference checkpoints ({n_ckpts}): {[label for label, _ in infer_checkpoints]}", flush=True)

n_test = len(sample_df)
prob_sum = np.zeros(n_test, dtype=np.float64)


@torch.inference_mode()
def run_checkpoint_infer(ckpt_path: Path, label: str) -> None:
    infer_model = timm.create_model(model_name, pretrained=False, num_classes=2)
    infer_model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    infer_model.to(device).eval()

    for imgs, indices in tqdm(test_loader, desc=f"Infer {label}", leave=False):
        batch_tensors, batch_map = [], []
        for j, img in enumerate(imgs):
            for view in tta_views(img, tta_n):
                batch_tensors.append(val_tf(view))
                batch_map.append(j)
        if not batch_tensors:
            continue
        x = torch.stack(batch_tensors).to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            batch_probs = torch.softmax(infer_model(x), dim=1)[:, 1].cpu().numpy()
        per_img = [[] for _ in range(len(imgs))]
        for prob, j in zip(batch_probs, batch_map):
            per_img[j].append(float(prob))
        for j, idx in enumerate(indices):
            prob_sum[idx] += float(np.mean(per_img[j]))

    del infer_model
    if device.type == "cuda":
        torch.cuda.empty_cache()


for label, path in tqdm(infer_checkpoints, desc="Checkpoints", leave=True):
    print(f"Running {label}: {path}", flush=True)
    run_checkpoint_infer(path, label)

probs = prob_sum / n_ckpts
preds = (probs >= threshold).astype(int)

infer_min = (time.perf_counter() - INFER_START) / 60.0
print(f"Inference done in {infer_min:.1f} min | positive rate: {100*preds.mean():.2f}%", flush=True)

## Bottle-type diagnostics
Per-type test prediction statistics (diagnostic only — global threshold unchanged).

In [ ]:
if CONFIG["use_bottletypes"] and bottletypes:
    diag_rows = [
        {
            "image_id": iid,
            "bottle_type": bottletypes.get(iid, "unknown"),
            "pred": p,
            "prob": pr,
        }
        for iid, p, pr in zip(sample_df["image_id"], preds, probs)
    ]
    diag_df = pd.DataFrame(diag_rows)
    print(f"{'bottle_type':<40} | {'n_images':>8} | {'pred_pos%':>10} | {'mean_prob':>9}", flush=True)
    print("-" * 75, flush=True)
    for btype, grp in diag_df.groupby("bottle_type", sort=False):
        print(
            f"{str(btype):<40} | {len(grp):8d} | {100*grp['pred'].mean():9.2f}% | {grp['prob'].mean():9.4f}",
            flush=True,
        )
    # To use per-type thresholds, replace global threshold with type_thresh dict from OOF analysis.
else:
    print("Bottle-type diagnostics skipped.", flush=True)

## Save submission
Write `submission.csv` with columns `[image_id, target]`.

In [ ]:
submission = sample_df[["image_id"]].copy()
submission["target"] = preds
output_path = Path(CONFIG["output_path"])
output_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(output_path, index=False)

print(f"Total predictions: {len(submission)}", flush=True)
print(f"Positive (reusable): {int(preds.sum())} ({100*preds.mean():.2f}%)", flush=True)
print(f"Saved: {output_path}", flush=True)

## Runtime summary
Total wall-clock time, peak GPU memory, and configuration used.

In [ ]:
elapsed_min = (time.perf_counter() - NOTEBOOK_START) / 60.0
elapsed_h = elapsed_min / 60.0
print(f"Training time:   {train_hours:.2f} h", flush=True)
print(f"Inference time:  {infer_min:.1f} min", flush=True)
print(f"Total elapsed:   {elapsed_h:.2f} h ({elapsed_min:.1f} min)", flush=True)

if device.type == "cuda":
    peak_gb = torch.cuda.max_memory_allocated(device) / 1024**3
    print(f"Peak GPU memory: {peak_gb:.2f} GB", flush=True)

print(f"Model: {model_name} | imgsz={IMGSZ} | epochs run <= {CONFIG['max_epochs']}", flush=True)
print(f"Checkpoints ensembled: {n_ckpts} | TTA views: {tta_n}", flush=True)
print(f"Best val F1: {best_f1:.4f} | Threshold: {threshold:.4f}", flush=True)

if elapsed_h > CONFIG["max_runtime_hours"]:
    print(
        f"Warning: exceeded {CONFIG['max_runtime_hours']} h budget — reduce max_epochs or use convnext_base.",
        flush=True,
    )
elif elapsed_h > CONFIG["max_runtime_hours"] * 0.9:
    print("Note: used >90% of 12 h budget — consider fewer epochs for Efficiency score.", flush=True)